# First Look at Hourly Wind Power in California and Wyoming

This notebook is a starting point for the Energy Meteorology project. It matches hourly capacity factors for one plant in California (CA) and one in Wyoming (WY), then compares their average daily patterns.

**Try it first:** Open this file in Jupyter or Google Colab and run the cells from top to bottom. The default demo uses **made-up data** to show how the notebook works. The demo is not a project result.

## 1. Choose the data

For real data, download one 2019 plant CSV from each state using the [PLUSWIND download page](https://wdh.energy.gov/ds/pluswind/plant.c0). Save local copies as `CA_2019.csv` and `WY_2019.csv` in `data/raw/`. Keep a record of the original filenames and plant IDs.

Then change `USE_DEMO = True` to `USE_DEMO = False` below and run all cells again. If you use Colab, upload the CSVs to the session and set the two paths below to match where you uploaded them.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

USE_DEMO = True       # Change to False when your two PLUSWIND CSVs are ready.
CA_FILE = Path("data/raw/CA_2019.csv")
WY_FILE = Path("data/raw/WY_2019.csv")
FIG_DIR = Path("figures")

TIME_COLUMN = "gmt"
TIME_FORMAT = "%Y%m%d%H"   # Check against the first rows of a real PLUSWIND file.
CF_COLUMN = "ERA5 CF (density and loss adjusted)"

# Create the expected folders (useful in a fresh clone or a Colab session).
CA_FILE.parent.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load and check the records

Capacity factor is the fraction of a plant's rated power. For example, 0.30 means 30%. The `gmt` column lets us compare both plants at the same moment, even though they are in different time zones.

In [ ]:
def read_plant(path, state):
    if not path.is_file():
        raise FileNotFoundError(
            f"Cannot find {path}. Check the file path in Step 1."
        )
    raw = pd.read_csv(path, encoding="utf-8-sig")
    raw.columns = raw.columns.str.strip()
    missing_columns = {TIME_COLUMN, CF_COLUMN} - set(raw.columns)
    if missing_columns:
        raise ValueError(
            f"{path.name} is missing: {', '.join(sorted(missing_columns))}. "
            f"Columns found: {list(raw.columns)}"
        )

    timestamp = pd.to_datetime(
        raw[TIME_COLUMN].astype("string").str.strip(),
        format=TIME_FORMAT,
        utc=True,
        errors="coerce",
    )
    if timestamp.isna().all():
        # Stop here instead of silently dropping every row.
        raise ValueError(
            f"{path.name}: no '{TIME_COLUMN}' value matches TIME_FORMAT={TIME_FORMAT!r}. "
            f"First value in the file: {raw[TIME_COLUMN].iloc[0]!r}. Update TIME_FORMAT in Step 1."
        )
    cf = pd.to_numeric(raw[CF_COLUMN], errors="coerce")
    plant = pd.DataFrame({"time_utc": timestamp, state: cf})
    valid = plant["time_utc"].notna() & plant[state].between(0, 1)
    print(f"{state}: {int((~valid).sum()):,} rows skipped (invalid time or capacity factor)")
    plant = plant.loc[valid]
    if plant["time_utc"].duplicated().any():
        raise ValueError(f"{path.name} contains duplicate hours. Check the input CSV.")
    return plant

if USE_DEMO:
    rng = np.random.default_rng(637)
    hours = pd.date_range("2019-01-01", periods=24 * 28, freq="h", tz="UTC")
    phase = 2 * np.pi * (hours.hour.to_numpy() - 6) / 24
    ca = pd.DataFrame({
        "time_utc": hours,
        "CA": np.clip(0.35 + 0.16 * np.cos(phase) + rng.normal(0, 0.07, len(hours)), 0, 1),
    })
    wy = pd.DataFrame({
        "time_utc": hours,
        "WY": np.clip(0.40 - 0.12 * np.cos(phase) + rng.normal(0, 0.09, len(hours)), 0, 1),
    })
    print("DEMO: Made-up data. Do not use these results in the project.")
else:
    ca = read_plant(CA_FILE, "CA")
    wy = read_plant(WY_FILE, "WY")

## 3. Match the same hours

Keep only hours that appear in both files. The checks below show how many valid hours were kept and how many are missing within the matched time span.

In [ ]:
paired = ca.merge(wy, on="time_utc", how="inner", validate="one_to_one")
paired = paired.sort_values("time_utc").reset_index(drop=True)
if paired.empty:
    raise ValueError("No valid hours overlap. Check that both files cover the same dates.")

expected = len(pd.date_range(paired["time_utc"].iloc[0], paired["time_utc"].iloc[-1], freq="h"))
print(f"Valid hours — CA: {len(ca):,}, WY: {len(wy):,}")
print(f"Matched hours: {len(paired):,}; gaps within matched span: {expected - len(paired):,}")
print(f"Date range (UTC): {paired['time_utc'].iloc[0]} to {paired['time_utc'].iloc[-1]}")
paired.head()

## 4. Plot the average daily cycle

For each hour of the day, average the matched values. This shows when each plant usually has higher or lower output in this sample. The combined line uses equal weights for this first test. The full project will use plant capacities to calculate the combined value.

The x-axis uses UTC so both plants are compared at the same moment. Local standard time is UTC−8 in California and UTC−7 in Wyoming (one hour earlier during daylight saving time). The figure is also saved to `figures/`.

In [ ]:
paired["Combined (equal weight)"] = (paired["CA"] + paired["WY"]) / 2
series = ["CA", "WY", "Combined (equal weight)"]
daily = paired.groupby(paired["time_utc"].dt.hour)[series].mean().reindex(range(24))

ax = daily.plot(figsize=(9, 4), marker="o", markersize=3)
ax.set(
    title="Average daily wind power pattern" + (" — DEMO" if USE_DEMO else ""),
    xlabel="Hour of day (UTC; CA local = UTC−8, WY local = UTC−7, standard time)",
    ylabel="Mean capacity factor",
    xticks=range(0, 24, 3),
    ylim=(0, 1),
)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(FIG_DIR / ("daily_cycle_demo.png" if USE_DEMO else "daily_cycle_2019.png"), dpi=200)
plt.show()

## 5. Make a first comparison

A negative correlation means one plant tends to be high when the other is low. Three correlations answer slightly different questions:

- **Hourly correlation** uses every matched hour, so it mixes daily, weather-system, and seasonal changes.
- **Diurnal-cycle correlation** compares the two 24-hour average curves from Step 4. This is the most direct test of whether the plants peak at different times of day.
- **Daily-anomaly correlation** removes each day's mean before correlating, so it focuses on changes within a day.

A smaller standard deviation means less variation in hourly capacity factor. Averaging two series always lowers the standard deviation unless they are perfectly correlated, so the combined value is compared with the **perfectly correlated reference**: the (equal-weight) average of the two individual standard deviations. The reduction relative to that reference is the smoothing that comes from the plants not moving together.

This first test only describes the two selected plants and the hours used here.

In [ ]:
if USE_DEMO:
    print("DEMO: Made-up data. Do not use these results in the project.")

def safe_corr(a, b):
    ok = a.notna() & b.notna()
    return a[ok].corr(b[ok]) if ok.sum() >= 3 else np.nan

# Remove each UTC day's mean to isolate within-day changes.
day = paired["time_utc"].dt.floor("D")
anom = paired[["CA", "WY"]] - paired.groupby(day)[["CA", "WY"]].transform("mean")

correlations = pd.Series({
    "Hourly": safe_corr(paired["CA"], paired["WY"]),
    "Diurnal cycle (24 hourly means)": safe_corr(daily["CA"], daily["WY"]),
    "Daily anomalies": safe_corr(anom["CA"], anom["WY"]),
}, name="CA–WY correlation")
print(correlations.round(2).to_string(), end="\n\n")

std = paired[series].std()
reference = (std["CA"] + std["WY"]) / 2   # Perfectly correlated, equal weights.
reduction = 1 - std["Combined (equal weight)"] / reference
print(f"Perfectly correlated reference std: {reference:.3f}")
print(f"Variability reduction vs. that reference: {reduction:.1%}")

std.rename("Hourly capacity factor standard deviation").round(3).to_frame()

## Next steps

Run this notebook on actual PLUSWIND files. Then add more plants and years, compare seasons, and check whether HRRR gives similar results. See [PROJECT_CHECKLIST.md](PROJECT_CHECKLIST.md) for the remaining work.